In [ ]:
import cv2
import os
from ultralytics import YOLO

# --- Load Your Model Once ---
model = YOLO('yolov8n-seg.pt')

# --- Define Folders ---
input_folder = "frames_input/"
output_folder_detect = "frames_output_detect/" # Folder for detection
output_folder_seg = "frames_output_seg/"     # Folder for segmentation

# Create the output folders if they don't exist
if not os.path.exists(output_folder_detect):
    os.makedirs(output_folder_detect)
if not os.path.exists(output_folder_seg):
    os.makedirs(output_folder_seg)

# Get all frame filenames, sorted in order
try:
    frame_files = sorted(os.listdir(input_folder))
    if not frame_files:
        print(f"Error: The folder '{input_folder}' is empty.")
        exit()
except FileNotFoundError:
    print(f"Error: The folder '{input_folder}' was not found.")
    exit()

print(f"Found {len(frame_files)} frames. Starting processing...")

# --- Loop Through Each Frame ---
for filename in frame_files:
    if not (filename.endswith(".png") or filename.endswith(".jpg")):
        continue

    frame_path = os.path.join(input_folder, filename)
    frame = cv2.imread(frame_path)

    if frame is None:
        print(f"Warning: Could not read frame {filename}. Skipping.")
        continue

    # --- Run Your Detection/Segmentation (Only Once) ---
    results = model.predict(frame, verbose=False)
    result = results[0]

    # --- !! TASK 2: Take Action !! ---
    action_text = "" # Text to draw if person is found
    if 0 in result.boxes.cls:
        print(f"ACTION: Person detected in frame {filename}!")
        action_text = "ACTION: Person Detected!"

    # --- Create TWO Processed Frames ---

    # 1. DETECTION FRAME (Boxes only, no masks)
    frame_detect = result.plot(masks=False)
    if action_text: # Add action text if needed
        cv2.putText(frame_detect, action_text, (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
    
    # 2. SEGMENTATION FRAME (Masks only, no boxes)
    frame_seg = result.plot(boxes=False)
    if action_text: # Add action text if needed
        cv2.putText(frame_seg, action_text, (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

    # --- Save Both Frames to Their Respective Folders ---
    output_path_detect = os.path.join(output_folder_detect, filename)
    cv2.imwrite(output_path_detect, frame_detect)
    
    output_path_seg = os.path.join(output_folder_seg, filename)
    cv2.imwrite(output_path_seg, frame_seg)

print(f"All frames processed!")
print(f"Detection frames saved to '{output_folder_detect}'.")
print(f"Segmentation frames saved to '{output_folder_seg}'.")